# 01 — First Look at the Data (Omar, S1.12)

**Draft: scope not yet confirmed with the team leader.**
This notebook takes a first look at the processed sessions: how many are
ready, their shapes, neurons per brain region, choice balance, block counts,
and the error and zero-contrast trials. It uses the team's modules
(`src.common.settings`, `src.analysis.eda`). No model is trained here.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the brain-decoding repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.eda import block_counts, region_counts, summarize_sessions
from src.common.settings import load_config, list_processed_sessions, load_session

config = load_config()
print("Project:", config["project_name"])
print("Feature window (seconds):", config["data"]["window"])

## 2. Which processed sessions are ready

In [ ]:
ready = list(list_processed_sessions())
print("Processed sessions available:", len(ready))


def demo_session():
    rng = np.random.default_rng(0)
    n_trials, n_neurons = 120, 12
    return {
        "X": rng.gamma(2.0, 2.0, (n_trials, n_neurons)),
        "y": rng.integers(0, 2, n_trials),
        "trial_number": np.arange(n_trials),
        "stim_side": rng.choice([-1, 1], n_trials),
        "contrast": rng.choice([0.0, 0.0625, 0.125, 0.25, 1.0], n_trials),
        "is_error": rng.random(n_trials) < 0.2,
        "is_zero_contrast": rng.random(n_trials) < 0.1,
        "prob_left": rng.choice([0.2, 0.5, 0.8], n_trials),
        "reaction_time": rng.uniform(0.1, 1.0, n_trials),
        "neuron_id": np.array([f"n{i}" for i in range(n_neurons)]),
        "neuron_region": np.array(["CA1"] * 5 + ["VISp"] * 4 + ["LGd"] * 3),
        "session_id": np.asarray("DEMO"),
    }


using_demo = len(ready) == 0
if using_demo:
    print("DEMO MODE: no processed sessions found yet.")
    print("Everything below uses a small synthetic session, NOT real data.")
    sessions = [demo_session()]
else:
    sessions = [load_session(sid) for sid in ready]

## 3. Summary of every session

In [ ]:
summary = summarize_sessions(sessions)
display(summary)

## 4. One session in detail

In [ ]:
data = sessions[0]
x = np.asarray(data["X"], dtype=float)
print("Session:", str(data["session_id"]), "(DEMO)" if using_demo else "")
print("X shape (trials x neurons):", x.shape)
print("Fields:", sorted(data))

### Neurons per brain region

In [ ]:
rc = region_counts(data)
display(rc.rename("neurons").to_frame())

fig, ax = plt.subplots(figsize=(8, 3.5))
rc.plot.bar(ax=ax)
ax.set_ylabel("Number of neurons")
ax.set_title("Neurons per brain region")
plt.tight_layout()
plt.show()

### Choice balance, block prior and control trials

In [ ]:
y = np.asarray(data["y"])
print("Trials:", len(y))
print("Choice balance (mean of y):", round(float(y.mean()), 3))
print("Error trials:", int(np.sum(data["is_error"])))
print("Zero-contrast trials:", int(np.sum(data["is_zero_contrast"])))
display(block_counts(data).rename("trials").to_frame())

### Firing rates

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(x.mean(axis=0), bins=20)
ax.set_xlabel("Mean firing rate per neuron (Hz)")
ax.set_ylabel("Number of neurons")
ax.set_title("Firing-rate distribution")
plt.tight_layout()
plt.show()

## Conclusions

No result is claimed in this notebook. The numbers above come from the run
of this notebook. If it ran in DEMO MODE, the tables use a synthetic session
and must not be reported. Re-run it after the processed sessions from S1.7
are available.